# 🚀 Run Gemma on Google Colab TPUs with TorchTPU - plain PyTorch, no CUDA

**What you'll do**
1. Verify the Colab **v5e-1 TPU** runtime and that `torch_tpu` gives you a `"tpu"` device
2. Load **Gemma 4 E2B (QAT, 4-bit repack)** from Hugging Face directly onto the TPU with `device_map="tpu"`
3. Build a fun use case: **🎓 Gemma Study Buddy** — paste any text, get a summary + a quiz
4. Benchmark `generate()` vs. a **single compiled decode step** (`torch.compile(backend="tpu")`)

**Before you start:** `Runtime → Change runtime type → v5e-1 TPU`, then `Runtime → Run all`.
No Hugging Face token needed for the model used here.

> Based on the TorchTPU + Gemma 4 E2B work by xbill9 ([article](https://dev.to/gde/pytorch-on-tpu-hands-on-with-colab-5b5g), [repo](https://github.com/xbill9/torchtpu-dev/tree/main/colab)). `torch_tpu` is preinstalled on the Colab TPU image. Versions in that article: torch_tpu 0.2.0, Transformers 5.20.x, compressed-tensors 0.19.0.

## 1️⃣ Check the TPU

In [1]:
import torch, importlib, sys

try:
    import torch_tpu
    print("torch_tpu:", getattr(torch_tpu, "__version__", "installed"))
except ImportError:
    sys.exit("torch_tpu not found. Set Runtime > Change runtime type > v5e-1 TPU and restart.")

print("torch:", torch.__version__)

# Tensors live on a 'tpu' device -- just like 'cuda'
x = torch.randn(2048, 2048, device="tpu", dtype=torch.bfloat16)
y = (x @ x).sum()
print("matmul ran on:", x.device, "| result:", y.item())

Successfully renamed PrivateUse1 backend to 'tpu'. Device: device(type='tpu')
Registered Python module for 'tpu'.
Device type: tpu, Device index: default
Initializing TPU distributed runtime


torch_tpu: installed
torch: 2.14.1+cpu
matmul ran on: tpu:0 | result: 1261568.0


## 2️⃣ Dependencies (only if the image is missing them)

In [2]:
import subprocess, sys
def have(pkg):
    try: importlib.import_module(pkg); return True
    except ImportError: return False

missing = [p for p, m in [("transformers","transformers"),("accelerate","accelerate"),
                          ("compressed-tensors","compressed_tensors")] if not have(m)]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
import transformers, compressed_tensors
print("transformers", transformers.__version__, "| compressed-tensors", compressed_tensors.__version__)

transformers 5.22.1 | compressed-tensors 0.20.1


## 3️⃣ Load Gemma straight onto the TPU

Two details matter (from the article's findings):
- `device_map="tpu"` puts **all** weights on the chip. `model.to("tpu")` alone left tensors on the host.
- This repack's tensor names are for the multimodal model (`model.language_model.*`), so `key_mapping` renames them for the text-only model.

In [3]:
import time, torch
from transformers import AutoTokenizer, AutoModelForCausalLM


In [4]:
MODEL_ID = "xbill9/gemma-4-E2B-it-qat-q4_0-w4a16-ct-text"

t0 = time.time()
tok = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    device_map="tpu",
    dtype=torch.bfloat16,
    key_mapping={r"^model\.language_model\.": "model."},
)
model.eval()
print(f"Loaded in {time.time()-t0:.1f}s")

# Sanity check: every tensor should be on the TPU
n_tpu = sum(p.device.type == "tpu" for p in model.parameters())
n_all = sum(1 for _ in model.parameters())
print(f"{n_tpu}/{n_all} parameters on TPU")

config.json:   0%|          | 0.00/2.97k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/3.73k [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 32.2MB            

tokenizer.json: downloading bytes:           |  0.00B            

chat_template.jinja:   0%|          | 0.00/18.6k [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/107k [00:00<?, ?B/s]

... (2 outputs left out)

Compressing model: 100%|██████████| 276/276 [00:00<00:00, 13224.29it/s]


Loading weights:   0%|          | 0/1092 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/203 [00:00<?, ?B/s]

Loaded in 80.2s
1057/1057 parameters on TPU


## 4️⃣ First generation

In [5]:
@torch.no_grad()
def chat(messages, max_new_tokens=200, **gen_kw):
    if isinstance(messages, str):
        messages = [{"role": "user", "content": messages}]
    enc = tok.apply_chat_template(messages, add_generation_prompt=True,
                                  return_tensors="pt", return_dict=True)
    enc = {k: v.to("tpu") for k, v in enc.items()}
    out = model.generate(**enc, max_new_tokens=max_new_tokens, do_sample=False, **gen_kw)
    new = out[0, enc["input_ids"].shape[1]:].cpu()
    return tok.decode(new, skip_special_tokens=True)

t0 = time.time()
print(chat("In two sentences, why are TPUs good for matrix multiplication?", max_new_tokens=80))
print(f"\n(first call includes compile time: {time.time()-t0:.1f}s)")

Decompressing model: 100%|██████████| 276/276 [00:04<00:00, 60.96it/s] 


TPUs are highly effective for matrix multiplication because they are specifically designed with massive parallelism and specialized hardware to handle the large number of simultaneous computations required by these operations. This architecture allows them to perform these calculations significantly faster and more efficiently than traditional CPUs or even GPUs for deep learning workloads.

(first call includes compile time: 29.2s)


## 5️⃣ Use case: 🎓 Gemma Study Buddy

Paste any text (lecture notes, an article, a README). Gemma, running on the TPU, returns a
**summary** and a **3-question quiz** as JSON, which we render as a collapsible quiz.

In [10]:
import json, re, html as _html
from IPython.display import HTML, display

INSTRUCTIONS = (
    "You are a study buddy. Read the source text below and respond with ONLY valid JSON "
    "(no markdown fences, no extra commentary) in exactly this shape:\n"
    '{"summary": "2-3 sentences", "quiz": [{"q": "question", "a": "answer"}, '
    '{"q": "question", "a": "answer"}, {"q": "question", "a": "answer"}]}'
)

def _extract_json(raw):
    """Pull the first {...} block out of the model output; return dict or None."""
    raw = re.sub(r"```(?:json)?", "", raw)           # strip fences if the model adds them
    m = re.search(r"\{.*\}", raw, re.S)
    if not m:
        return None
    try:
        return json.loads(m.group(0))
    except json.JSONDecodeError:
        return None

def study_buddy(text, max_new_tokens=500):
    # Instructions go in the user turn (works with any chat template, with or without a system role)
    prompt = f"{INSTRUCTIONS}\n\nSOURCE TEXT:\n{text.strip()}"
    raw = chat(prompt, max_new_tokens=max_new_tokens)
    data = _extract_json(raw)
    quiz = data.get("quiz") if isinstance(data, dict) else None
    if not data or not data.get("summary") or not isinstance(quiz, list) or not quiz:
        print("Could not parse a valid summary/quiz. Raw output:\n", raw)
        return None
    esc = lambda s: _html.escape(str(s))
    out = f"<h3>📝 Summary</h3><p>{esc(data['summary'])}</p><h3>❓ Quiz</h3><ol>"
    for item in quiz:
        out += (f"<li><b>{esc(item.get('q', '?'))}</b>"
                f"<details><summary>Show answer</summary>{esc(item.get('a', '?'))}</details></li>")
    display(HTML(out + "</ol>"))
    return data

NOTES = """
Tensor Processing Units (TPUs) are custom accelerators designed by Google for machine learning.
Their core is the matrix multiply unit (MXU), a systolic array that streams data through a grid of
multiply-accumulate cells, minimizing memory traffic. TPUs favor bfloat16, which keeps float32's
exponent range but uses fewer mantissa bits, so training stays stable at half the memory cost.
Programs are compiled ahead of time by the XLA compiler, which fuses many operations into one kernel.
Because compiled programs are specialized to tensor shapes, changing input shapes triggers recompilation.
"""
_ = study_buddy(NOTES)

## 6️⃣ Benchmark: plain `generate()` (eager)
TorchTPU supports eager, defer-and-fuse and `torch.compile`. The article measured `generate()` at ~5 tok/s on v5e-1 because
the sequence grows each step and every new length is a new shape. Your numbers may differ.

In [9]:
def bench_generate(prompt="Explain how a transformer works.", n=64):
    chat(prompt, max_new_tokens=4)                      # warm-up
    t0 = time.time(); out = chat(prompt, max_new_tokens=n); dt = time.time() - t0
    ntok = len(tok(out)["input_ids"])
    print(f"generate(): ~{ntok/dt:.1f} tok/s  ({ntok} tokens in {dt:.1f}s)")
bench_generate()

generate(): ~4.7 tok/s  (64 tokens in 13.6s)


## 7️⃣ 🔥 Advanced: one compiled decode step (the big speedup)

Idea from the article: use a **fixed-shape `StaticCache`**, remove compressed-tensors' per-module forward wrappers
(which cause ~24 graph breaks), and `torch.compile(backend="tpu")` a *single-token step*. The article reports roughly
**31–33 tok/s vs ~5 tok/s** for `generate()` on the same chip.

⚠️ Notes: experimental; context limited to the cache size (256 here, within the 512-token sliding window);
the first call compiles for ~40 s. The article reports that a static cache past the sliding window halted the TPU — keep `CACHE_LEN` ≤ 512.

In [13]:
from compressed_tensors.offload.dispatch import remove_dispatch
from transformers import StaticCache
import transformers.cache_utils as cu

remove_dispatch(model)   # drop the wrapped forwards so torch.compile can trace one graph

# Sliding-window layers count length in a Python int (-> recompiles). Use the plain static layer's methods instead.
for m in ("update", "get_seq_length", "get_mask_sizes"):
    setattr(cu.StaticSlidingWindowLayer, m, getattr(cu.StaticLayer, m))

CACHE_LEN = 256
cache = StaticCache(config=model.config, max_cache_len=CACHE_LEN)

def move_cpu_to_tpu(root, verbose=False, max_depth=8):
    """Find tensors still on the CPU inside `root` (cache layers, model modules, ...) and move them to the TPU.
    TorchTPU's compiled graphs need EVERY input on the TPU; one stray CPU scalar raises
    'expected tensor to be on tpu, got cpu'."""
    seen, moved = set(), []
    def fix(v, path):
        if isinstance(v, torch.Tensor) and v.device.type == "cpu":
            moved.append((path, tuple(v.shape), str(v.dtype)))
            return v.to("tpu")
        return v
    def walk(o, path, depth):
        if id(o) in seen or depth > max_depth or isinstance(o, (str, bytes, int, float, bool, type(None), torch.Tensor)):
            return
        seen.add(id(o))
        if isinstance(o, dict):
            for k in list(o.keys()):
                o[k] = fix(o[k], f"{path}[{k!r}]"); walk(o[k], f"{path}[{k!r}]", depth + 1)
        elif isinstance(o, list):
            for i in range(len(o)):
                o[i] = fix(o[i], f"{path}[{i}]"); walk(o[i], f"{path}[{i}]", depth + 1)
        elif isinstance(o, tuple):
            for i, x in enumerate(o): walk(x, f"{path}[{i}]", depth + 1)
        elif hasattr(o, "__dict__"):
            for k, v in list(vars(o).items()):
                nv = fix(v, f"{path}.{k}")
                if nv is not v: setattr(o, k, nv) if not isinstance(o, torch.nn.Module) else o.__dict__.__setitem__(k, nv)
                walk(nv, f"{path}.{k}", depth + 1)
    walk(root, type(root).__name__, 0)
    if verbose or moved:
        print(f"moved {len(moved)} CPU tensor(s) to TPU", *[f"  {p} {s} {d}" for p, s, d in moved[:15]], sep="\n")
    return len(moved)

@torch.no_grad()
def _step(tok_ids):
    logits = model(input_ids=tok_ids, past_key_values=cache, use_cache=True, logits_to_keep=1).logits
    return logits[:, -1].argmax(-1)

# The cache is created lazily on the first forward pass, so run ONE eager step to build it,
# then move any CPU-side tensors (e.g. position counters) onto the TPU before compiling.
_ = _step(torch.tensor([[tok.bos_token_id or 0]], device="tpu"))
cache.reset()
move_cpu_to_tpu(model, verbose=True)
move_cpu_to_tpu(cache, verbose=True)

step = torch.compile(_step, backend="tpu", dynamic=False)

@torch.no_grad()
def fast_generate(prompt, max_new_tokens=100):
    enc = tok.apply_chat_template([{"role": "user", "content": prompt}], add_generation_prompt=True,
                                  return_tensors="pt", return_dict=True)
    ids = enc["input_ids"][0].tolist()
    assert len(ids) + max_new_tokens <= CACHE_LEN, "prompt + new tokens must fit in CACHE_LEN"
    cache.reset()
    move_cpu_to_tpu(cache)   # reset() can re-create counters on the CPU
    # Feed the prompt one token at a time (fixed shape -> one compiled program)
    for t in ids:
        nxt = step(torch.tensor([[t]], device="tpu"))
    out = []
    for _ in range(max_new_tokens):          # tokens stay on the chip; no per-token host readback
        out.append(nxt)
        nxt = step(nxt.view(1, 1))
    toks = torch.cat(out).cpu().tolist()     # single readback at the end
    eos = tok.eos_token_id
    cut = [i for i, t in enumerate(toks) if t in ([eos] if isinstance(eos, int) else eos or [])]
    return tok.decode(toks[: cut[0]] if cut else toks, skip_special_tokens=True)

t0 = time.time(); fast_generate("Hi!", 8); print(f"compile + first call: {time.time()-t0:.1f}s")

moved 0 CPU tensor(s) to TPU
moved 0 CPU tensor(s) to TPU
compile + first call: 16.7s


In [14]:
prompt = "Write a short haiku-style explanation of what a TPU is."
t0 = time.time(); out = fast_generate(prompt, max_new_tokens=100); dt = time.time() - t0
print(out)
print(f"\n~{100/dt:.1f} tok/s (includes feeding the prompt; 100 decode steps)")

Matrix of chips,
Learning patterns, fast and deep,
AI's brain grows.



~34.1 tok/s (includes feeding the prompt; 100 decode steps)


## 🧠 Takeaways
- Same PyTorch / Hugging Face code, `device_map="tpu"` instead of `"cuda"`. No driver setup.
- **Shapes decide compile count** on TPU: fixed shapes (static cache, bucketed lengths) are the key to speed.
- Streaming each token back to the host caps throughput; collect on-chip and read back at the end when you can.

**Ideas to extend:** swap the Study Buddy for a code reviewer, or a long-context recall test (see the article's ring-cache notebook 05).